In [6]:
# ==============================================================================
# Cell 1: Load and Split the Dataset
# ==============================================================================
import json
import os

# --- Configuration ---
# 1. Make sure to upload your dataset to Colab's file system first.
# 2. Change this filename to match your uploaded file.
original_dataset_filename = '/kaggle/input/prepare-dataset/prepare_Dataset.json' # <-- IMPORTANT: Change if your filename is different

# Define the output filenames
knowledge_base_filename = 'knowledge_base.json'
validation_set_filename = 'validation_set.json'

# --- Main Logic ---
print("--- Step 1: Data Splitting ---")

# Check if the original dataset file exists
if not os.path.exists(original_dataset_filename):
    print(f"❌ ERROR: The file '{original_dataset_filename}' was not found.")
    print("Please make sure you have uploaded the dataset and the filename is correct.")
else:
    # Load the entire dataset from the JSON file
    with open(original_dataset_filename, 'r') as f:
        all_data = json.load(f)
    print(f"✅ Successfully loaded {len(all_data)} total records.")

    # Calculate the split point (80% of the data)
    total_size = len(all_data)
    split_point = int(total_size * 0.8)

    # Split the data sequentially (no shuffling, as requested)
    knowledge_base_data = all_data[:split_point]
    validation_set_data = all_data[split_point:]

    # Verify the sizes
    print(f"Splitting data...")
    print(f"  - Knowledge Base size (80%): {len(knowledge_base_data)} records")
    print(f"  - Validation Set size (20%): {len(validation_set_data)} records")

    # Save the knowledge base to a new JSON file
    with open(knowledge_base_filename, 'w') as f:
        json.dump(knowledge_base_data, f, indent=4)
    print(f"✅ Knowledge Base saved to '{knowledge_base_filename}'")

    # Save the validation set to a new JSON file
    with open(validation_set_filename, 'w') as f:
        json.dump(validation_set_data, f, indent=4)
    print(f"✅ Validation Set saved to '{validation_set_filename}'")

    print("\n--- Data split complete. ---")

--- Step 1: Data Splitting ---
✅ Successfully loaded 2100 total records.
Splitting data...
  - Knowledge Base size (80%): 1680 records
  - Validation Set size (20%): 420 records
✅ Knowledge Base saved to 'knowledge_base.json'
✅ Validation Set saved to 'validation_set.json'

--- Data split complete. ---


In [ ]:
!pip3 install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
!pip install numpy==1.26.4 sentence-transformers==2.7.0 chromadb==0.5.0 tqdm --upgrade transformers accelerate bitsandbytes -q

In [8]:
import json
import torch
from sentence_transformers import SentenceTransformer
import chromadb

# --- 2. Configuration ---
embedding_model_name = 'BAAI/bge-large-en-v1.5'
knowledge_base_filename = 'knowledge_base.json'
validation_set_filename = 'validation_set.json'
validation_with_embeddings_filename = 'validation_set_with_embeddings.json'
collection_name = "bdd_scenarios_kb"

# --- 3. Load Embedding Model ---
print("--- Initializing Models and Database ---")

# Check for GPU availability and set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Using device: {device}")

# Load the sentence-transformer model. This may take a minute to download.
print(f"⏳ Loading embedding model '{embedding_model_name}'...")
embedding_model = SentenceTransformer(embedding_model_name, device=device)
print("✅ Embedding model loaded successfully.")

# --- 4. Setup Chroma DB ---
# Initialize the Chroma DB client. We'll use an in-memory/on-disk version.
client = chromadb.Client()
try:
    # We try to delete the old collection if it exists
    client.delete_collection(name=collection_name)
    print(f"🗑️ Deleted existing collection '{collection_name}' to prevent duplicates.")
except Exception:
    # If the collection didn't exist yet, we just skip this part
    print(f"🆕 No existing collection '{collection_name}' found. Creating a fresh one.")
# Create a new collection. If it already exists, this will load the existing one.
collection = client.get_or_create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"} # Use cosine similarity
)
print(f"✅ Chroma DB collection '{collection_name}' is ready.")
print("\n--- Setup complete. Proceed to the next cell to index the knowledge base. ---")

--- Initializing Models and Database ---
✅ Using device: cuda
⏳ Loading embedding model 'BAAI/bge-large-en-v1.5'...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Embedding model loaded successfully.
🗑️ Deleted existing collection 'bdd_scenarios_kb' to prevent duplicates.
✅ Chroma DB collection 'bdd_scenarios_kb' is ready.

--- Setup complete. Proceed to the next cell to index the knowledge base. ---


In [9]:
# ==============================================================================
# Cell 3: Index the Knowledge Base (Part 2a)
# ==============================================================================
from tqdm.auto import tqdm # For nice progress bars

print("--- Part 2a: Processing and Indexing the Knowledge Base ---")

# Load the knowledge base data
with open(knowledge_base_filename, 'r') as f:
    knowledge_base_data = json.load(f)

# Prepare lists for batch processing
ids_to_add = []
records_to_embed = []
metadata_to_add = []

print("⏳ Preparing knowledge base records...")
for i, item in enumerate(knowledge_base_data):
    # Dynamically find the keys for the record and feature
    record_key = next((key for key in item if key.startswith('Record')), None)
    feature_key = next((key for key in item if key.startswith('Feature')), None)

    if record_key and feature_key:
        record_text = item[record_key]
        bdd_scenario_text = item[feature_key]

        # Add data to our lists for batch processing
        records_to_embed.append(bdd_scenario_text)
        ids_to_add.append(f"kb_record_{i}")
        metadata_to_add.append({
            "original_record": record_text,
            "bdd_scenario": bdd_scenario_text
        })

# Create embeddings in a single batch (much faster)
print(f"⏳ Creating embeddings for {len(records_to_embed)} knowledge base records... (This may take a moment)")
embeddings_to_add = embedding_model.encode(
    records_to_embed,
    batch_size=32, # Adjust batch size based on your GPU memory
    show_progress_bar=True,
    convert_to_tensor=False
)

# Add the data to Chroma DB in one go
print("⏳ Adding all records to Chroma DB collection...")
collection.add(
    ids=ids_to_add,
    embeddings=embeddings_to_add.tolist(), # Convert numpy array to list
    metadatas=metadata_to_add
)

print(f"✅ Successfully indexed {collection.count()} records into the knowledge base.")
print("\n--- Knowledge base indexing complete. ---")

--- Part 2a: Processing and Indexing the Knowledge Base ---
⏳ Preparing knowledge base records...
⏳ Creating embeddings for 1680 knowledge base records... (This may take a moment)


Batches:   0%|          | 0/53 [00:00<?, ?it/s]

⏳ Adding all records to Chroma DB collection...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


✅ Successfully indexed 1680 records into the knowledge base.

--- Knowledge base indexing complete. ---


In [10]:
# ==============================================================================
# Cell 4: Prepare the Validation Set (Part 2b)
# ==============================================================================
from tqdm.auto import tqdm

print("--- Part 2b: Processing the Validation Set ---")

# Load the validation set data
with open(validation_set_filename, 'r') as f:
    validation_set_data = json.load(f)

validation_records_to_embed = []
enhanced_validation_data = []

print("⏳ Preparing validation set records...")
for item in validation_set_data:
    record_key = next((key for key in item if key.startswith('Record')), None)
    feature_key = next((key for key in item if key.startswith('Feature')), None)

    if record_key and feature_key:
        record_text = item[record_key]
        bdd_scenario_text = item[feature_key]
        validation_records_to_embed.append(bdd_scenario_text)
        # Store the plain text data in our new structure
        enhanced_validation_data.append({
            "scenario_text": bdd_scenario_text,
            "golden_record": record_text
        })

# Create embeddings for the validation set in a single batch
print(f"⏳ Creating embeddings for {len(validation_records_to_embed)} validation records...")
validation_embeddings = embedding_model.encode(
    validation_records_to_embed,
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=False
)

# Add the newly created embeddings to our enhanced data structure
for i, item in enumerate(enhanced_validation_data):
    item['embedding'] = validation_embeddings[i].tolist()

# Save the processed validation set to a new file
with open(validation_with_embeddings_filename, 'w') as f:
    json.dump(enhanced_validation_data, f, indent=4)

print(f"✅ Validation set processed and saved with embeddings to '{validation_with_embeddings_filename}'")
print("\n--- Validation set preparation complete. ---")

--- Part 2b: Processing the Validation Set ---
⏳ Preparing validation set records...
⏳ Creating embeddings for 420 validation records...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

✅ Validation set processed and saved with embeddings to 'validation_set_with_embeddings.json'

--- Validation set preparation complete. ---


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
user_secrets = UserSecretsClient()
hf_token= user_secrets.get_secret("HF_demo")


# Log in to Hugging Face
login(token=hf_token)

In [ ]:
import torch
import transformers

model_id = "meta-llama/Llama-3.1-8B"

print(f"--- Loading Llama 3.1 model into a pipeline: {model_id} ---")
print("This will take several minutes and use a significant amount of RAM...")

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    # CHANGE THIS LINE: Use float16 for T4/P100 compatibility
    model_kwargs={"torch_dtype": torch.float16}, 
    device_map="auto"
)
print("✅ Llama 3.1 pipeline ready.")

In [ ]:
# ==============================================================================
# Cell 1: Master Installation (Force Upgrade for Security Fix)
# ==============================================================================
# This upgrades PyTorch to fix the 'CVE-2025-32434' ValueError.

!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install numpy==1.26.4 sentence-transformers==2.7.0 chromadb==0.5.0 transformers==4.42.3 accelerate bitsandbytes tqdm evaluate -q

print("✅ Libraries updated to PyTorch 2.6+!")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_id = "Salesforce/codet5p-770m"

print(f"--- Loading model: {model_id} ---")

# 1. Load Tokenizer explicitly
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    trust_remote_code=True
)

# 2. Load Model explicitly
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Create Pipeline using the loaded model/tokenizer
pipeline = pipeline(
    "text2text-generation", 
    model=model, 
    tokenizer=tokenizer
)

print("✅ CodeT5+ pipeline ready.")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_id = "Salesforce/codet5p-220m"

print(f"--- Loading model: {model_id} ---")

# 1. Load Tokenizer explicitly
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    trust_remote_code=True
)

# 2. Load Model explicitly
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Create Pipeline using the loaded model/tokenizer
pipeline = pipeline(
    "text2text-generation", 
    model=model, 
    tokenizer=tokenizer
)

print("✅ CodeT5+ pipeline ready.")

Deepseek

In [ ]:
#Ensure u restart
!pip install -U bitsandbytes

In [1]:
# ==============================================================================
# Optimized Model Loading Script (for DeepSeek Coder 33B with Quantization)
# ==============================================================================
# This script uses 4-bit quantization with bitsandbytes to load the large
# 33B model efficiently on the available Kaggle GPUs.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig

model_id = "deepseek-ai/deepseek-coder-33b-base"

print(f"--- Loading DeepSeek Coder 33B Instruct model with 4-bit quantization: {model_id} ---")
print("This will still take several minutes, but will be much more memory-efficient.")

# --- THE OPTIMIZATION: Configuration for 4-bit quantization ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16 # Use float16 for T4 compatibility
)
# -------------------------------------------------------------

# --- V1 models REQUIRE trust_remote_code=True ---
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True
)

# --- Load the model with the quantization config ---
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    quantization_config=bnb_config, # <-- APPLIED THE QUANTIZATION
    device_map="auto" # This will now place the smaller model entirely on your GPUs
)
# --------------------------------------------------

pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)
tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

print("✅ DeepSeek Coder 33B Instruct pipeline ready (loaded in 4-bit).")

2026-01-13 02:05:24.915394: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768269924.936145     302 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768269924.942557     302 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768269924.959037     302 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768269924.959056     302 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768269924.959058     302 computation_placer.cc:177] computation placer alr

--- Loading DeepSeek Coder 33B Instruct model with 4-bit quantization: deepseek-ai/deepseek-coder-33b-base ---
This will still take several minutes, but will be much more memory-efficient.


tokenizer_config.json:   0%|          | 0.00/793 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

...

model-00005-of-00007.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Device set to use cuda:0


✅ DeepSeek Coder 33B Instruct pipeline ready (loaded in 4-bit).


In [ ]:
pip install evaluate scikit-learn

In [27]:
# ='============================================================================
# Final Evaluation Script (with Smart Sequential Checkpointing)
# ==============================================================================
# This script automatically finds the latest checkpoint (e.g., checkpoint_2.json)
# and saves the next batch of progress to a new file (e.g., checkpoint_3.json).

import json
from tqdm.auto import tqdm
import evaluate
import numpy as np
import gc
import torch
import os
import re # Added for finding numbers in filenames

# --- 1. Smart Checkpoint Helper Function ---
def get_checkpoint_paths():
    """Finds the latest checkpoint file and determines the next one."""
    latest_n = -1
    input_checkpoint = None
    
    # Find all files matching the checkpoint pattern
    for filename in os.listdir('.'):
        if filename.startswith('evaluation_records_checkpoint') and filename.endswith('.json'):
            # Use regex to find a number in the filename, e.g., checkpoint_3.json -> 3
            match = re.search(r'_(\d+)\.json', filename)
            if match:
                n = int(match.group(1))
                if n > latest_n:
                    latest_n = n
                    input_checkpoint = filename
    
    # If no numbered checkpoint is found, check for the original un-numbered one
    if latest_n == -1 and os.path.exists('evaluation_records_checkpoint.json'):
        input_checkpoint = 'evaluation_records_checkpoint.json'
        output_checkpoint = 'evaluation_records_checkpoint_1.json'
    else:
        # Create the next filename in the sequence (e.g., _1 -> _2)
        output_checkpoint = f'evaluation_records_checkpoint_{latest_n + 1}.json'
        # If there were no numbered files found, this correctly starts with _0.json if needed,
        # but our logic below handles the very first run separately.
        
    return input_checkpoint, output_checkpoint

# --- 2. Required Helper Functions ---

def build_prompt(task_scenario, retrieved_examples):
    prompt = """1.Your role:
You are an expert test analyst.
2.Your task:
Your task is to translate the given "BDD Scenario" into a compliant and accurate “Record text”. Use the provided examples to understand the correct format and style.
After you have completed the Record text for the Task, you must stop. Do not generate any further examples or text.
"""
    for i, example in enumerate(retrieved_examples):
        record, scenario = example['original_record'], example['bdd_scenario']
        example_dict = {"Source BDD Scenario": scenario, "Record": record}
        prompt += f"\nExample{i+1}:\n{json.dumps(example_dict, indent=4)}\n"

    task_dict = {"Source BDD Scenario": task_scenario}
    task_json_str = json.dumps(task_dict, indent=4).strip()[:-1].strip()
    prompt += f"""
Task:
{task_json_str},
    "Record": """
    return prompt

def calculate_metrics(predictions, references, tokenizer):
    print("\n--- Calculating Performance Metrics ---")
    em_score = evaluate.load("exact_match").compute(references=references, predictions=predictions)['exact_match']
    bleu_score = evaluate.load("bleu").compute(predictions=[p if p.strip() else " " for p in predictions], references=[[r] for r in references])['bleu']
    f1_scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens, ref_tokens = set(tokenizer.encode(pred)), set(tokenizer.encode(ref))
        common_tokens = pred_tokens.intersection(ref_tokens)
        if not common_tokens: f1_scores.append(0.0); continue
        precision = len(common_tokens) / len(pred_tokens) if pred_tokens else 0
        recall = len(common_tokens) / len(ref_tokens) if ref_tokens else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_scores.append(f1)
    f1_score = np.mean(f1_scores)
    return {"f1_score": f1_score, "bleu_score": bleu_score, "exact_match_score": em_score}

# --- 3. Load Full Validation Data ---
with open('validation_set_with_embeddings.json', 'r') as f:
    full_validation_data = json.load(f)

# --- 4. Smart Checkpointing Setup ---
input_checkpoint, output_checkpoint = get_checkpoint_paths()
report_data = []

if input_checkpoint:
    print(f"🔄 Found latest checkpoint: '{input_checkpoint}'. Resuming progress.")
    with open(input_checkpoint, 'r') as f:
        report_data = json.load(f)
    start_index = len(report_data)
else:
    print(f"🏁 Starting a new evaluation run. First save will be 'evaluation_records_checkpoint.json'.")
    # If no file exists at all, the first save will be the un-numbered one
    output_checkpoint = 'evaluation_records_checkpoint.json'
    start_index = 0

records_to_process = full_validation_data[start_index:]
print(f"--- Starting FULL EVALUATION for {len(records_to_process)} remaining records ---")
print(f"--- New progress will be saved to '{output_checkpoint}' ---")

# ==============================================================================
# THE KEY CHANGE IS HERE: The Main Evaluation Loop with RegEx Cleanup
# ==============================================================================
if records_to_process:
    for i, item in enumerate(tqdm(records_to_process, desc="Evaluating Full Set")):
        
        results = collection.query(query_embeddings=[item['embedding']], n_results=1) #Testing
        final_prompt = build_prompt(item['scenario_text'], results['metadatas'][0])
        
        outputs = pipeline(final_prompt, max_new_tokens=512, eos_token_id=pipeline.tokenizer.eos_token_id, do_sample=False, pad_token_id=pipeline.tokenizer.eos_token_id)

        # This is the raw output from the model, starting AFTER our prompt ends
        generated_part_raw = outputs[0]['generated_text'][len(final_prompt):].strip()  #For llama model
        #generated_part_raw = outputs[0]['generated_text'].strip() #For codet5p model
        
        # --- BULLETPROOF REGEX CLEANUP ---
        generated_record = "" # Default to an empty string if extraction fails
        
        # This regex looks for text between the first quote and the last quote
        # that is followed by a closing brace '}'.
        match = re.search(r'^\s*"(.*)"\s*\}', generated_part_raw, re.DOTALL)
        
        if match:
            # If a match is found, extract the clean text
            clean_raw_text = match.group(1)
            try:
                # Un-escape characters like \n and \"
                generated_record = clean_raw_text.encode().decode('unicode_escape')
            except Exception:
                # As a fallback, use the raw extracted text
                generated_record = clean_raw_text
        else:
            # If the regex fails, it means the output was malformed.
            # We log a warning and use the raw text as a fallback.
            print(f"Warning: Could not extract Record for item {start_index + i}. Using raw output.")
            generated_record = generated_part_raw
        # --------------------------------

        report_data.append({
            "input_scenario": item['scenario_text'],
            "expected_record": item['golden_record'],
            "generated_record": generated_record
        })

        del results, final_prompt, outputs
        gc.collect()
        torch.cuda.empty_cache()

        # Save progress to the NEW (n+1) checkpoint file.
        if (i + 1) % 25 == 0 or (i + 1) == len(records_to_process):
            with open(output_checkpoint, 'w') as f:
                json.dump(report_data, f, indent=4)
            tqdm.write(f"💾 Checkpoint saved to '{output_checkpoint}'. Total records: {len(report_data)}")

print("✅ Full evaluation run complete.")

# --- 6. Calculate Metrics ---
predictions = [d['generated_record'] for d in report_data]
references = [d['expected_record'] for d in report_data]
metrics = calculate_metrics(predictions, references, pipeline.tokenizer)

# --- 7. Generate and Save the Final Detailed Report ---
report_filename = "final_evaluation_report_records.txt"
report_lines = []
for data in report_data:
    report_lines.append("Input BDD Scenario:\n")
    report_lines.append(data['input_scenario'])
    report_lines.append("\n\nExpected Record:\n")
    report_lines.append(data['expected_record'])
    report_lines.append("\n\nGenerated Record:\n")
    report_lines.append(data['generated_record'])
    report_lines.append("\n" + "="*50 + "\n")

report_lines.append("\n. . .\n\n")
report_lines.append(f"F1 Score (Token-based): {metrics['f1_score']:.4f}")
report_lines.append(f"BLEU Score:               {metrics['bleu_score']:.4f}")
report_lines.append(f"Exact Match Score:        {metrics['exact_match_score']:.4f}")

final_report = "\n".join(report_lines)
with open(report_filename, 'w') as f:
    f.write(final_report)

print("\n--- FINAL REPORT SUMMARY ---")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"BLEU Score: {metrics['bleu_score']:.4f}")
print(f"Exact Match: {metrics['exact_match_score']:.4f}")
print(f"\n✅ Detailed report saved to '{report_filename}'")

🏁 Starting a new evaluation run. First save will be 'evaluation_records_checkpoint.json'.
--- Starting FULL EVALUATION for 420 remaining records ---
--- New progress will be saved to 'evaluation_records_checkpoint.json' ---


Evaluating Full Set:   0%|          | 0/420 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
For testing

In [ ]:
# ='============================================================================
# Final Evaluation Script (with Smart Sequential Checkpointing)
# ==============================================================================
# This script automatically finds the latest checkpoint (e.g., checkpoint_2.json)
# and saves the next batch of progress to a new file (e.g., checkpoint_3.json).

import json
from tqdm.auto import tqdm
import evaluate
import numpy as np
import gc
import torch
import os
import re # Added for finding numbers in filenames

# --- 1. Smart Checkpoint Helper Function ---
def get_checkpoint_paths():
    """Finds the latest checkpoint file and determines the next one."""
    latest_n = -1
    input_checkpoint = None
    
    # Find all files matching the checkpoint pattern
    for filename in os.listdir('.'):
        if filename.startswith('evaluation_records_checkpoint') and filename.endswith('.json'):
            # Use regex to find a number in the filename, e.g., checkpoint_3.json -> 3
            match = re.search(r'_(\d+)\.json', filename)
            if match:
                n = int(match.group(1))
                if n > latest_n:
                    latest_n = n
                    input_checkpoint = filename
    
    # If no numbered checkpoint is found, check for the original un-numbered one
    if latest_n == -1 and os.path.exists('evaluation_records_checkpoint.json'):
        input_checkpoint = 'evaluation_records_checkpoint.json'
        output_checkpoint = 'evaluation_records_checkpoint_1.json'
    else:
        # Create the next filename in the sequence (e.g., _1 -> _2)
        output_checkpoint = f'evaluation_records_checkpoint_{latest_n + 1}.json'
        # If there were no numbered files found, this correctly starts with _0.json if needed,
        # but our logic below handles the very first run separately.
        
    return input_checkpoint, output_checkpoint

# --- 2. Required Helper Functions ---

# --- FIX: The build_prompt function is now simplified for a single example ---
def build_prompt(task_scenario, retrieved_example): # It now takes one example, not a list
    prompt = """1.Your role:
You are an expert test analyst.
2.Your task:
Your task is to translate the given "Source BDD Scenario" into a compliant and accurate “Record text”. Use the provided examples to understand the correct format and style.
"""
    
    # Use simple, human-readable markdown headers for the single example.
    record = retrieved_example['original_record']
    scenario = retrieved_example['bdd_scenario']
    prompt += f"\n--- Example 1 ---\n"
    prompt += f"Source BDD Scenario:\n{scenario}\n"
    prompt += f"\nRecord:\n{record}\n"
    
    # The final task is also a simple markdown block.
    prompt += "\n--- Task ---\n"
    prompt += f"Source BDD Scenario:\n{task_scenario}\n"
    prompt += "\nRecord:\n"
    
    return prompt

def calculate_metrics(predictions, references, tokenizer):
    print("\n--- Calculating Performance Metrics ---")
    em_score = evaluate.load("exact_match").compute(references=references, predictions=predictions)['exact_match']
    bleu_score = evaluate.load("bleu").compute(predictions=[p if p.strip() else " " for p in predictions], references=[[r] for r in references])['bleu']
    f1_scores = []
    for pred, ref in zip(predictions, references):
        pred_tokens, ref_tokens = set(tokenizer.encode(pred)), set(tokenizer.encode(ref))
        common_tokens = pred_tokens.intersection(ref_tokens)
        if not common_tokens: f1_scores.append(0.0); continue
        precision = len(common_tokens) / len(pred_tokens) if pred_tokens else 0
        recall = len(common_tokens) / len(ref_tokens) if ref_tokens else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_scores.append(f1)
    f1_score = np.mean(f1_scores)
    return {"f1_score": f1_score, "bleu_score": bleu_score, "exact_match_score": em_score}

# --- 3. Load Full Validation Data ---
with open('validation_set_with_embeddings.json', 'r') as f:
    full_validation_data = json.load(f)

# --- 4. Smart Checkpointing Setup ---
input_checkpoint, output_checkpoint = get_checkpoint_paths()
report_data = []

if input_checkpoint:
    print(f"🔄 Found latest checkpoint: '{input_checkpoint}'. Resuming progress.")
    with open(input_checkpoint, 'r') as f:
        report_data = json.load(f)
    start_index = len(report_data)
else:
    print(f"🏁 Starting a new evaluation run. First save will be 'evaluation_records_checkpoint.json'.")
    # If no file exists at all, the first save will be the un-numbered one
    output_checkpoint = 'evaluation_records_checkpoint.json'
    start_index = 0

records_to_process = full_validation_data[start_index:]
print(f"--- Starting FULL EVALUATION for {len(records_to_process)} remaining records ---")
print(f"--- New progress will be saved to '{output_checkpoint}' ---")

# ==============================================================================
# THE KEY CHANGE IS HERE: The Main Evaluation Loop with RegEx Cleanup
# ==============================================================================
if records_to_process:
    for i, item in enumerate(tqdm(records_to_process, desc="Evaluating Full Set")):
        
        results = collection.query(query_embeddings=[item['embedding']], n_results=1) #Testing
        final_prompt = build_prompt(item['scenario_text'], results['metadatas'][0][0])
        
        outputs = pipeline(final_prompt, max_new_tokens=512, eos_token_id=pipeline.tokenizer.eos_token_id, do_sample=False, pad_token_id=pipeline.tokenizer.eos_token_id)

        # This is the raw output from the model, starting AFTER our prompt ends
        generated_part_raw = outputs[0]['generated_text'][len(final_prompt):].strip()  #For llama model
        #generated_part_raw = outputs[0]['generated_text'].strip() #For codet5p model
        
        # --- BULLETPROOF REGEX CLEANUP ---
        generated_record = "" # Default to an empty string if extraction fails
        
        # This regex looks for text between the first quote and the last quote
        # that is followed by a closing brace '}'.
        match = re.search(r'^\s*"(.*)"\s*\}', generated_part_raw, re.DOTALL)
        
        if match:
            # If a match is found, extract the clean text
            clean_raw_text = match.group(1)
            try:
                # Un-escape characters like \n and \"
                generated_record = clean_raw_text.encode().decode('unicode_escape')
            except Exception:
                # As a fallback, use the raw extracted text
                generated_record = clean_raw_text
        else:
            # If the regex fails, it means the output was malformed.
            # We log a warning and use the raw text as a fallback.
            print(f"Warning: Could not extract Record for item {start_index + i}. Using raw output.")
            generated_record = generated_part_raw
        # --------------------------------

        report_data.append({
            "input_scenario": item['scenario_text'],
            "expected_record": item['golden_record'],
            "generated_record": generated_record
        })

        del results, final_prompt, outputs
        gc.collect()
        torch.cuda.empty_cache()

        # Save progress to the NEW (n+1) checkpoint file.
        if (i + 1) % 25 == 0 or (i + 1) == len(records_to_process):
            with open(output_checkpoint, 'w') as f:
                json.dump(report_data, f, indent=4)
            tqdm.write(f"💾 Checkpoint saved to '{output_checkpoint}'. Total records: {len(report_data)}")

print("✅ Full evaluation run complete.")

# --- 6. Calculate Metrics ---
predictions = [d['generated_record'] for d in report_data]
references = [d['expected_record'] for d in report_data]
metrics = calculate_metrics(predictions, references, pipeline.tokenizer)

# --- 7. Generate and Save the Final Detailed Report ---
report_filename = "final_evaluation_report_records.txt"
report_lines = []
for data in report_data:
    report_lines.append("Input BDD Scenario:\n")
    report_lines.append(data['input_scenario'])
    report_lines.append("\n\nExpected Record:\n")
    report_lines.append(data['expected_record'])
    report_lines.append("\n\nGenerated Record:\n")
    report_lines.append(data['generated_record'])
    report_lines.append("\n" + "="*50 + "\n")

report_lines.append("\n. . .\n\n")
report_lines.append(f"F1 Score (Token-based): {metrics['f1_score']:.4f}")
report_lines.append(f"BLEU Score:               {metrics['bleu_score']:.4f}")
report_lines.append(f"Exact Match Score:        {metrics['exact_match_score']:.4f}")

final_report = "\n".join(report_lines)
with open(report_filename, 'w') as f:
    f.write(final_report)

print("\n--- FINAL REPORT SUMMARY ---")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"BLEU Score: {metrics['bleu_score']:.4f}")
print(f"Exact Match: {metrics['exact_match_score']:.4f}")
print(f"\n✅ Detailed report saved to '{report_filename}'")

In [ ]:
rm /kaggle/working/final_evaluation_report_records.txt

In [ ]:
rm /kaggle/working/evaluation_records_checkpoint.json

In [25]:
rm /kaggle/working/evaluation_records_checkpoint_2.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [23]:
rm /kaggle/working/state.db

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [22]:
alias rm=trash

ValueError: not enough values to unpack (expected 2, got 1)

In [15]:
rm /kaggle/working/evaluation_records_checkpoint_2.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [16]:
!mv /kaggle/input/temp-file/evaluation_records_checkpoint_2.json /kaggle/working/

mv: cannot remove '/kaggle/input/temp-file/evaluation_records_checkpoint_2.json': Read-only file system


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [12]:
rm /kaggle/working/evaluation_records_checkpoint.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [13]:
rm /kaggle/working/evaluation_records_checkpoint_1.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [5]:
rm /kaggle/working/knowledge_base.json

In [4]:
rm /kaggle/working/validation_set.json

In [3]:
rm /kaggle/working/validation_set_with_embeddings.json

In [ ]:
rm -rf /kaggle/working/*